# HADA Full-Mamba No-RA — BCI Competition IV-2b LOSO (S01-S09)
Enable **GPU** and **Internet** in Kaggle, then run the cell below.


In [ ]:
# Full LOSO over all nine BCI IV-2b subjects.
import shutil
import socket
import subprocess
import sys
import time

def require_kaggle_internet():
    hosts = ('github.com', 'pypi.org', 'download.pytorch.org')
    last_error = None
    for attempt in range(1, 4):
        try:
            for host in hosts:
                socket.getaddrinfo(host, 443)
            print('Kaggle Internet/DNS check passed.')
            return
        except OSError as exc:
            last_error = exc
            print(f'Internet check {attempt}/3 failed: {exc}', flush=True)
            time.sleep(5)
    raise RuntimeError(
        'Kaggle Internet is unavailable. Open Notebook options, set Internet=On, '
        'save the session, and run again. Last DNS error: ' + str(last_error)
    )

require_kaggle_internet()
if shutil.which('uv') is None:
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', '--retries', '2', '--timeout', '30', 'uv'],
        check=True,
    )
else:
    print('Using preinstalled uv:', shutil.which('uv'))

BRANCH = 'feature/hada-full-mamba-no-ra'
%cd /kaggle/working
!if [ -d tcformer-test/.git ]; then git -C tcformer-test remote set-url origin https://github.com/CuongDM1806/tcformer-test.git && git -C tcformer-test fetch origin $BRANCH && git -C tcformer-test checkout --force $BRANCH && git -C tcformer-test reset --hard origin/$BRANCH; else git clone --branch $BRANCH --single-branch https://github.com/CuongDM1806/tcformer-test.git; fi
%cd /kaggle/working/tcformer-test
!git log -1 --oneline
!grep -nE 'all_subject_ids = list\(range\(1, 10\)\)|dataset="2b"|riemannian_alignment: false|im_tta_steps: 5' datamodules/bcic4_2b.py configs/hada_tcformer.yaml
!grep -nE 'ModelCheckpoint|monitor=|best_model_path|adapt_to_target\(adaptation_loader\)' train_pipeline.py
!echo 'Verified: best-validation Full-Mamba with IM-TTA and RA disabled.'
!uv venv --clear --python 3.10 .venv
!uv pip install --python .venv/bin/python torch==2.7.1 torchvision==0.22.1 --index-url https://download.pytorch.org/whl/cu126
!uv pip install --python .venv/bin/python -r requirements.txt
!.venv/bin/python -c "import yaml; from pathlib import Path; p=Path('configs/hada_tcformer.yaml'); c=yaml.safe_load(p.read_text()); c['preprocessing']['bcic2b']['riemannian_alignment']=False; c['model_kwargs']['im_tta_steps']=5; p.write_text(yaml.safe_dump(c, sort_keys=False))"
!nvidia-smi
!PYTHONUNBUFFERED=1 MPLBACKEND=Agg .venv/bin/python -u train_pipeline.py --model hada_tcformer --dataset bcic2b --loso --gpu_id 0
